In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib qt

def read_and_plot(sim_file, title):
    # Read the entire file and parse multiple step data
    with open(sim_file, 'r') as f:
        lines = f.readlines()
    
    # The first line should be the header
    header_line = lines[0].strip()
    if not (header_line.startswith('VCC') and 'V(rc)' in header_line and 'I(RC)' in header_line):
        print(f"Warning: Expected header not found. Found: {header_line}")
        return pd.DataFrame()
    
    # Find all step information and data sections
    steps_data = []
    vbb_values = []
    
    i = 1  # Start after header
    current_step_data = []
    current_vbb = None
    
    while i < len(lines):
        line = lines[i].strip()
        
        # Look for step information
        if line.startswith('Step Information:') and 'VBB=' in line:
            # Save previous step data if exists
            if current_step_data and current_vbb is not None:
                # Create dataframe for previous step
                from io import StringIO
                data_str = header_line + '\n' + '\n'.join(current_step_data)
                step_df = pd.read_csv(StringIO(data_str), sep=r'\s+')
                step_df.columns = step_df.columns.str.strip()
                step_df['VBB'] = current_vbb
                steps_data.append(step_df)
                vbb_values.append(current_vbb)
            
            # Extract VBB value for new step
            vbb_str = line.split('VBB=')[1].split()[0]
            current_vbb = float(vbb_str)
            current_step_data = []
            
        elif line and not line.startswith('Step Information:'):
            # This is data line
            current_step_data.append(line)
        
        i += 1
    
    # Don't forget the last step
    if current_step_data and current_vbb is not None:
        from io import StringIO
        data_str = header_line + '\n' + '\n'.join(current_step_data)
        step_df = pd.read_csv(StringIO(data_str), sep=r'\s+')
        step_df.columns = step_df.columns.str.strip()
        step_df['VBB'] = current_vbb
        steps_data.append(step_df)
        vbb_values.append(current_vbb)
    
    if not steps_data:
        print("No data found in file")
        return pd.DataFrame()
    
    print(f"Found {len(steps_data)} steps with VBB values: {vbb_values}")
    
    # Create plots with multiple curves for different VBB values
    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 10))
    
    colors = ['tab:blue', 'tab:red', 'tab:green', 'tab:orange', 'tab:purple']
    
    # Plot V(RC) for each VBB value
    for idx, (step_df, vbb) in enumerate(zip(steps_data, vbb_values)):
        color = colors[idx % len(colors)]
        ax[0].plot(step_df['VCC'], step_df['V(rc)'], 
                  label=f'VBB = {vbb}V', color=color, linewidth=2)
    
    ax[0].set_ylabel('V(RC) (V)')
    ax[0].set_title('Collector Resistor Voltage vs VCC')
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Plot I(RC) for each VBB value (convert to mA)
    for idx, (step_df, vbb) in enumerate(zip(steps_data, vbb_values)):
        color = colors[idx % len(colors)]
        ax[1].plot(step_df['VCC'], step_df['I(RC)'] * 1000, 
                  label=f'VBB = {vbb}V', color=color, linewidth=2)
    
    ax[1].set_xlabel('VCC (V)')
    ax[1].set_ylabel('I(RC) (mA)')
    ax[1].set_title('Collector Resistor Current vs VCC')
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    # Return combined dataframe
    if steps_data:
        combined_df = pd.concat(steps_data, ignore_index=True)
        return combined_df
    else:
        return pd.DataFrame()

# Example usage
read_and_plot('/home/artj/Documents/ECE3201CircII/labs/lab04/idealnpn.txt', 'Ideal NPN Transistor DC Sweep Analysis (VCC vs VBB parametric)')
read_and_plot('/home/artj/Documents/ECE3201CircII/labs/lab04/circuitsim2n3904.txt', '2N3904 Transistor DC Sweep Analysis (VCC vs VBB parametric)')

QApplication: invalid style override 'kvantum' passed, ignoring it.
	Available styles: Windows, Fusion


Found 3 steps with VBB values: [9.5, 17.4, 25.3]
Found 3 steps with VBB values: [9.5, 17.4, 25.3]


,VCC,V(rc),I(RC),VBB
0,0.0,0.003820,-0.000004,9.5
1,0.1,0.028674,0.000071,9.5
2,0.2,0.043106,0.000157,9.5
3,0.3,0.053232,0.000247,9.5
4,0.4,0.061096,0.000339,9.5
...,...,...,...,...
298,9.6,0.729654,0.008870,25.3
299,9.7,0.821508,0.008878,25.3
300,9.8,0.913363,0.008887,25.3
301,9.9,1.005217,0.008895,25.3


In [2]:
# Install required packages and import libraries


# Data Analysis and Comparison
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats

# Load and store the data from both transistor types
ideal_data = read_and_plot('/home/artj/Documents/ECE3201CircII/labs/lab04/idealnpn.txt', 'Ideal NPN Analysis')
real_data = read_and_plot('/home/artj/Documents/ECE3201CircII/labs/lab04/circuitsim2n3904.txt', '2N3904 Analysis')

# Add transistor type column for comparison
ideal_data['Transistor_Type'] = 'Ideal NPN'
real_data['Transistor_Type'] = '2N3904'

# Combine datasets
combined_data = pd.concat([ideal_data, real_data], ignore_index=True)

print("=== TRANSISTOR COMPARISON ANALYSIS ===\n")

# 1. Basic Statistics Summary
print("1. BASIC STATISTICS COMPARISON")
print("=" * 50)

for transistor in ['Ideal NPN', '2N3904']:
    data_subset = combined_data[combined_data['Transistor_Type'] == transistor]
    print(f"\n{transistor} Transistor:")
    print(f"  - Data points: {len(data_subset)}")
    print(f"  - VCC range: {data_subset['VCC'].min():.1f}V to {data_subset['VCC'].max():.1f}V")
    print(f"  - VBB levels: {sorted(data_subset['VBB'].unique())}")
    print(f"  - V(RC) range: {data_subset['V(rc)'].min():.3f}V to {data_subset['V(rc)'].max():.3f}V")
    print(f"  - I(RC) range: {data_subset['I(RC)'].min()*1000:.3f}mA to {data_subset['I(RC)'].max()*1000:.3f}mA")

# 2. Saturation Analysis
print("\n\n2. SATURATION CHARACTERISTICS")
print("=" * 50)

def find_saturation_point(group, threshold=0.01):
    """Find where current change becomes less than threshold (saturation)"""
    current_diff = np.diff(group['I(RC)'])
    try:
        sat_idx = np.where(np.abs(current_diff) < threshold/1000)[0][0]  # Convert mA to A
        return group.iloc[sat_idx]['VCC']
    except:
        return np.nan

saturation_analysis = combined_data.groupby(['Transistor_Type', 'VBB']).apply(find_saturation_point).reset_index()
saturation_analysis.columns = ['Transistor_Type', 'VBB', 'Saturation_VCC']

print("Saturation Voltage (VCC where current plateaus):")
pivot_sat = saturation_analysis.pivot(index='VBB', columns='Transistor_Type', values='Saturation_VCC')
print(pivot_sat.round(2))

# 3. Current Gain Analysis (Beta estimation)
print("\n\n3. CURRENT GAIN ANALYSIS")
print("=" * 50)

# Calculate maximum collector current for each VBB level
max_currents = combined_data.groupby(['Transistor_Type', 'VBB'])['I(RC)'].max().reset_index()
max_currents['I(RC)_mA'] = max_currents['I(RC)'] * 1000

# Estimate base current (assuming VBE ≈ 0.7V and typical base resistor)
# This is an approximation since we don't have base current directly
max_currents['Estimated_IB_mA'] = (max_currents['VBB'] - 0.7) / 10000  # Assuming 10kΩ base resistor
max_currents['Estimated_Beta'] = max_currents['I(RC)_mA'] / max_currents['Estimated_IB_mA']

print("Estimated Current Gain (β = IC/IB):")
beta_pivot = max_currents.pivot(index='VBB', columns='Transistor_Type', values='Estimated_Beta')
print(beta_pivot.round(1))

# 4. Statistical Tests
print("\n\n4. STATISTICAL ANALYSIS")
print("=" * 50)

# Compare V(RC) distributions
ideal_vrc = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']['V(rc)']
real_vrc = combined_data[combined_data['Transistor_Type'] == '2N3904']['V(rc)']

# Perform t-test
t_stat, p_value = stats.ttest_ind(ideal_vrc, real_vrc)
print(f"T-test for V(RC) differences:")
print(f"  - T-statistic: {t_stat:.3f}")
print(f"  - P-value: {p_value:.2e}")
print(f"  - Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(ideal_vrc)-1)*ideal_vrc.var() + (len(real_vrc)-1)*real_vrc.var()) / 
                     (len(ideal_vrc) + len(real_vrc) - 2))
cohens_d = (ideal_vrc.mean() - real_vrc.mean()) / pooled_std
print(f"  - Effect size (Cohen's d): {cohens_d:.3f}")

# 5. Performance Metrics
print("\n\n5. PERFORMANCE METRICS")
print("=" * 50)

performance_metrics = combined_data.groupby(['Transistor_Type', 'VBB']).agg({
    'I(RC)': ['max', 'std'],
    'V(rc)': ['max', 'std'],
    'VCC': 'count'
}).round(4)

print("Performance Summary (max current, voltage stability):")
print(performance_metrics)

# 6. Key Differences Summary
print("\n\n6. KEY DIFFERENCES SUMMARY")
print("=" * 50)

ideal_max_current = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']['I(RC)'].max() * 1000
real_max_current = combined_data[combined_data['Transistor_Type'] == '2N3904']['I(RC)'].max() * 1000

ideal_max_voltage = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']['V(rc)'].max()
real_max_voltage = combined_data[combined_data['Transistor_Type'] == '2N3904']['V(rc)'].max()

print(f"Maximum Collector Current:")
print(f"  - Ideal NPN: {ideal_max_current:.3f} mA")
print(f"  - 2N3904:    {real_max_current:.3f} mA")
print(f"  - Difference: {abs(ideal_max_current - real_max_current):.3f} mA")

print(f"\nMaximum Collector Voltage:")
print(f"  - Ideal NPN: {ideal_max_voltage:.3f} V")
print(f"  - 2N3904:    {real_max_voltage:.3f} V") 
print(f"  - Difference: {abs(ideal_max_voltage - real_max_voltage):.3f} V")

# Current efficiency comparison
ideal_efficiency = (ideal_max_current / 10) * 100  # Current per volt VCC
real_efficiency = (real_max_current / 10) * 100
print(f"\nCurrent Efficiency (mA per volt VCC):")
print(f"  - Ideal NPN: {ideal_efficiency:.2f}%")
print(f"  - 2N3904:    {real_efficiency:.2f}%")

print(f"\n{'='*60}")
print("CONCLUSION: The analysis reveals key differences in saturation")
print("characteristics, current gain, and overall performance between")
print("the ideal and real transistor models.")
print(f"{'='*60}")

Found 3 steps with VBB values: [9.5, 17.4, 25.3]
Found 3 steps with VBB values: [9.5, 17.4, 25.3]
=== TRANSISTOR COMPARISON ANALYSIS ===

1. BASIC STATISTICS COMPARISON

Ideal NPN Transistor:
  - Data points: 303
  - VCC range: 0.0V to 10.0V
  - VBB levels: [np.float64(9.5), np.float64(17.4), np.float64(25.3)]
  - V(RC) range: 0.007V to 8.994V
  - I(RC) range: -0.012mA to 2.999mA

2N3904 Transistor:
  - Data points: 303
  - VCC range: 0.0V to 10.0V
  - VBB levels: [np.float64(9.5), np.float64(17.4), np.float64(25.3)]
  - V(RC) range: 0.004V to 6.750V
  - I(RC) range: -0.005mA to 8.903mA


2. SATURATION CHARACTERISTICS
Saturation Voltage (VCC where current plateaus):
Transistor_Type  2N3904  Ideal NPN
VBB                               
9.5                 3.4        1.3
17.4                6.3        2.3
25.3                9.2        3.3


3. CURRENT GAIN ANALYSIS
Estimated Current Gain (β = IC/IB):
Transistor_Type  2N3904  Ideal NPN
VBB                               
9.5              

/tmp/ipykernel_55064/4013901950.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  saturation_analysis = combined_data.groupby(['Transistor_Type', 'VBB']).apply(find_saturation_point).reset_index()


In [ ]:
# Advanced Visualizations for Transistor Comparison
plt.style.use('seaborn-v0_8')
fig = plt.figure(figsize=(16, 12))

# Create a 2x3 subplot layout
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Side-by-side comparison of I(RC) vs VCC
ax1 = fig.add_subplot(gs[0, :])
for vbb in sorted(combined_data['VBB'].unique()):
    ideal_subset = combined_data[(combined_data['Transistor_Type'] == 'Ideal NPN') & (combined_data['VBB'] == vbb)]
    real_subset = combined_data[(combined_data['Transistor_Type'] == '2N3904') & (combined_data['VBB'] == vbb)]
    
    ax1.plot(ideal_subset['VCC'], ideal_subset['I(RC)'] * 1000, 
             label=f'Ideal (VBB={vbb}V)', linestyle='-', linewidth=2)
    ax1.plot(real_subset['VCC'], real_subset['I(RC)'] * 1000, 
             label=f'2N3904 (VBB={vbb}V)', linestyle='--', linewidth=2)

ax1.set_xlabel('VCC (V)')
ax1.set_ylabel('I(RC) (mA)')
ax1.set_title('Collector Current Comparison: Ideal vs 2N3904')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Distribution comparison - Current
ax2 = fig.add_subplot(gs[1, 0])
ideal_currents = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']['I(RC)'] * 1000
real_currents = combined_data[combined_data['Transistor_Type'] == '2N3904']['I(RC)'] * 1000

ax2.hist(ideal_currents, bins=30, alpha=0.7, label='Ideal NPN', density=True)
ax2.hist(real_currents, bins=30, alpha=0.7, label='2N3904', density=True)
ax2.set_xlabel('I(RC) (mA)')
ax2.set_ylabel('Density')
ax2.set_title('Current Distribution Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Distribution comparison - Voltage
ax3 = fig.add_subplot(gs[1, 1])
ideal_voltages = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']['V(rc)']
real_voltages = combined_data[combined_data['Transistor_Type'] == '2N3904']['V(rc)']

ax3.hist(ideal_voltages, bins=30, alpha=0.7, label='Ideal NPN', density=True)
ax3.hist(real_voltages, bins=30, alpha=0.7, label='2N3904', density=True)
ax3.set_xlabel('V(RC) (V)')
ax3.set_ylabel('Density')
ax3.set_title('Voltage Distribution Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Scatter plot: Current vs Voltage relationship
ax4 = fig.add_subplot(gs[2, 0])
ideal_data_plot = combined_data[combined_data['Transistor_Type'] == 'Ideal NPN']
real_data_plot = combined_data[combined_data['Transistor_Type'] == '2N3904']

scatter1 = ax4.scatter(ideal_data_plot['V(rc)'], ideal_data_plot['I(RC)'] * 1000, 
                      c=ideal_data_plot['VBB'], cmap='viridis', alpha=0.6, 
                      marker='o', s=20, label='Ideal NPN')
scatter2 = ax4.scatter(real_data_plot['V(rc)'], real_data_plot['I(RC)'] * 1000, 
                      c=real_data_plot['VBB'], cmap='plasma', alpha=0.6, 
                      marker='^', s=20, label='2N3904')

ax4.set_xlabel('V(RC) (V)')
ax4.set_ylabel('I(RC) (mA)')
ax4.set_title('I-V Characteristic Scatter Plot')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter1, ax=ax4)
cbar.set_label('VBB (V)')

# 5. Performance summary bar chart
ax5 = fig.add_subplot(gs[2, 1])
metrics_data = combined_data.groupby('Transistor_Type').agg({
    'I(RC)': 'max',
    'V(rc)': 'max'
}).reset_index()

x = np.arange(len(metrics_data))
width = 0.35

bars1 = ax5.bar(x - width/2, metrics_data['I(RC)'] * 1000, width, 
                label='Max Current (mA)', alpha=0.8)
ax5_twin = ax5.twinx()
bars2 = ax5_twin.bar(x + width/2, metrics_data['V(rc)'], width, 
                     label='Max Voltage (V)', alpha=0.8, color='orange')

ax5.set_xlabel('Transistor Type')
ax5.set_ylabel('Max Current (mA)', color='blue')
ax5_twin.set_ylabel('Max Voltage (V)', color='orange')
ax5.set_title('Maximum Performance Comparison')
ax5.set_xticks(x)
ax5.set_xticklabels(metrics_data['Transistor_Type'])

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}', ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax5_twin.text(bar.get_x() + bar.get_width()/2., height,
                  f'{height:.2f}', ha='center', va='bottom')

plt.suptitle('Output Difference Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics table
print("\n" + "="*80)
print("COMPREHENSIVE SUMMARY TABLE")
print("="*80)

summary_stats = combined_data.groupby('Transistor_Type').agg({
    'I(RC)': ['mean', 'std', 'min', 'max'],
    'V(rc)': ['mean', 'std', 'min', 'max'],
    'VCC': 'count'
}).round(4)

# Convert current to mA for better readability
current_cols = [col for col in summary_stats.columns if 'I(RC)' in col[0]]
for col in current_cols:
    summary_stats[col] = summary_stats[col] * 1000

summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns]
print(summary_stats)

print(f"\n{'='*80}")
print("KEY INSIGHTS:")
print("• Ideal transistor shows more linear behavior")
print("• 2N3904 exhibits more realistic saturation characteristics") 
print("• Both models show similar trends but different magnitudes")
print("• Real transistor has non-idealities that affect performance")
print(f"{'='*80}")


COMPREHENSIVE SUMMARY TABLE
                 I(RC)_mean  I(RC)_std  I(RC)_min  I(RC)_max  V(rc)_mean  \
Transistor_Type                                                            
2N3904                  3.9        2.3       -0.0        8.9      1.1393   
Ideal NPN               1.7        0.9       -0.0        3.0      3.2628   

                 V(rc)_std  V(rc)_min  V(rc)_max  VCC_count  
Transistor_Type                                              
2N3904              1.7263     0.0038     6.7501        303  
Ideal NPN           2.6660     0.0070     8.9935        303  

KEY INSIGHTS:
• Ideal transistor shows more linear behavior
• 2N3904 exhibits more realistic saturation characteristics
• Both models show similar trends but different magnitudes
• Real transistor has non-idealities that affect performance


/tmp/ipykernel_55064/2674005144.py:108: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
